PROJECT TITLE-SpendDNA - Financial Fingerprint Analysis

NAME: Sri Durga Varsini P

# SpendDNA – Financial Fingerprint Analysis

## Objective

SpendDNA analyses six months of synthetic Indian UPI/bank
transactions to identify spending patterns, vendors,
categories, monthly trends, anomalies and spending archetypes.

Dataset: Rahul Sharma's synthetic transaction data
Period: January 2024 – June 2024

In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("rahul_transactions.csv")

df.head()


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [15]:
df.shape




(1328, 8)

In [16]:
df.columns

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')

In [17]:
df['Date'] = pd.to_datetime(
    df['Date'],
    errors='coerce',
    dayfirst=True
)
df['Date'].head()

,Date
0,2024-01-01
1,NaT
2,NaT
3,2024-01-01
4,NaT


In [18]:
df['Date'].head(20)

,Date
0,2024-01-01
1,NaT
2,NaT
3,2024-01-01
4,NaT
5,2024-01-01
6,2024-01-01
7,NaT
8,2024-02-01
9,NaT


In [19]:
df['Date'].dtype

dtype('<M8[ns]')

In [20]:
 df_raw = pd.read_csv("rahul_transactions.csv")
 df_raw['Date'].head(20)

,Date
0,2024-01-01
1,01-Jan-24
2,01-Jan-24
3,2024-01-01
4,01 Jan 2024
5,2024-01-01
6,2024-01-01
7,01-Jan-24
8,2024-01-02
9,02/01/24


In [21]:
df_raw['Date'].unique()[:20]

array(['2024-01-01', '01-Jan-24', '01 Jan 2024', '2024-01-02', '02/01/24',
       '02-Jan-24', '02 Jan 2024', '03-Jan-24', '03/01/24', '2024-01-03',
       '04-Jan-24', '2024-01-04', '04 Jan 2024', '04/01/24', '05-Jan-24',
       '05/01/24', '2024-01-05', '06 Jan 2024', '06/01/24', '06-Jan-24'],
      dtype=object)

In [22]:
df = pd.read_csv("rahul_transactions.csv")

In [23]:
df['Date'] = pd.to_datetime(
    df['Date'],
    dayfirst=True,
    errors='coerce',
    format='mixed'
)
print("Missing dates:", df['Date'].isna().sum())

Missing dates: 0


In [24]:
df['Date'].head(20)

,Date
0,2024-01-01
1,2024-01-01
2,2024-01-01
3,2024-01-01
4,2024-01-01
5,2024-01-01
6,2024-01-01
7,2024-01-01
8,2024-01-02
9,2024-01-02


In [25]:
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
)

df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
print("Missing amounts:", df['Amount'].isna().sum())

Missing amounts: 0


In [26]:
df['Amount'].head(20)

,Amount
0,2462.0
1,50.0
2,84728.0
3,1828.0
4,270.0
5,625.0
6,148.0
7,482.0
8,537.0
9,3956.0


In [27]:
df['Type'] = (
    df['Type']
    .astype(str)
    .str.strip()
    .str.upper()
)

df['Type'] = df['Type'].replace({
    'DR': 'Debit',
    'CR': 'Credit',
    'DEBIT': 'Debit',
    'CREDIT': 'Credit'
})
df['Type'].value_counts()
print("Duplicates before:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicates after:", df.duplicated().sum())
print("Total transactions:", len(df))

Duplicates before: 18
Duplicates after: 0
Total transactions: 1310


In [28]:
print("Total transactions:", len(df))
print("Missing dates:", df['Date'].isna().sum())
print("Missing amounts:", df['Amount'].isna().sum())
print("Duplicates:", df.duplicated().sum())
print("Transaction types:")
print(df['Type'].value_counts())

Total transactions: 1310
Missing dates: 0
Missing amounts: 0
Duplicates: 0
Transaction types:
Type
Debit     1304
Credit       6
Name: count, dtype: int64


## Feature 2: Vendor Extractor

In [29]:
## Feature 2: Vendor Extractor
print("Total unique descriptions:")
print(df['Description'].nunique())

print("\nSample descriptions:")
print(df['Description'].unique()[:30])


Total unique descriptions:
283

Sample descriptions:
['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT'
 'GROFERS INDIA P L' 'POS UBER BANGALORE' 'BANGALORE ELEC SUPPLY'
 'TWC INDIA' 'UPI-BESCOM-BILL@HDFCBANK' 'UPI-AMAN-0816@OKAXIS'
 'ROPPEN TRANSPORTATION' 'OLA CABS' 'POS ZOMATO' 'UPI-AMAZONPAY@HDFCBANK']


In [30]:
vendor_keywords = {
    "Amazon": ["AMAZON", "AMZN"],
    "Blinkit": ["BLINKIT", "GROFERS"],
    "Zepto": ["ZEPTO"],
    "Swiggy Instamart": ["SWIGGY-INSTAMART"],
    "Swiggy": ["SWIGGY"],
    "Zomato": ["ZOMATO"],
    "Uber": ["UBER"],
    "Ola Electric": ["OLA ELECTRIC"],
    "Ola": ["OLA CABS", "OLA-PRIME", "ANI TECHNOLOGIES"],
    "BMTC": ["BMTC"],
    "Groww": ["GROWW"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRDWAVE", "TWC INDIA"],
    "Truffles": ["TRUFFLES"],
    "Flipkart": ["FLIPKART"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC SUPPLY"],
    "BookMyShow": ["BMS MOVIE"],
    "Rapido": ["ROPPEN"],
    "TechCrush Labs": ["TECHCRUSH LABS"],
    "Aman": ["UPI-AMAN"]
}


In [31]:
def extract_vendor(description):
    description = str(description).upper()

    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in description:
                return vendor

    return "Unknown"

In [32]:
df["vendor_clean"] = df["Description"].apply(extract_vendor)

df[["Description", "vendor_clean"]].head(20)

,Description,vendor_clean
0,AMAZON SELLER SVCS,Amazon
1,BHIM-BMTC,BMTC
2,NEFT-TECHCRUSH LABS-SALARY MAY24,TechCrush Labs
3,UPI-AMAN-8934@OKAXIS,Aman
4,BHIM-BLINKIT,Blinkit
5,BHIM ZEPTO,Zepto
6,UPI-UBER-2426@HDFCBANK,Uber
7,BHIM-BLINKIT,Blinkit
8,POS SWIGGY BANGALORE,Swiggy
9,UPI-GROWWPAY@HDFCBANK,Groww


In [33]:
print("Unknown transactions:")
print((df["vendor_clean"] == "Unknown").sum())

print("\nUnidentified descriptions:")
print(
    df.loc[
        df["vendor_clean"] == "Unknown",
        "Description"
    ].unique()
)

Unknown transactions:
442

Unidentified descriptions:
['IMPS-RENT-LANDLORD-75500265' 'UPI-ANKIT-6430@OKAXIS'
 'UPI-OLACABS@HDFCBANK' 'UPI-JIORECHARGE@PAYTM' 'UPI-CCD@HDFCBANK'
 'INSTAMART BANGALORE' 'AVENUE SUPERMARTS' 'POS HP PETROL STATION'
 'UPI-VIKAS-6060@OKAXIS' 'POS BANGALORE RESTAURANT'
 'UPI-ZERODHA-COIN@AXIS' 'UPI-BOOKMYSHOW@HDFCBANK' 'UPI-DMART@OKAXIS'
 'UPI-RAPIDO@OKAXIS' 'BUNDL Tech P L' 'POS CAFE COFFEE DAY'
 'UPI-MYNTRA@HDFCBANK' 'UPI-BWSSB@AXIS' 'UPI-NETFLIX-MEMB@HDFCBANK'
 'AIRTEL POSTPAID' 'UPI-PRIYA-2221@OKAXIS' 'POS NYKAA BANGALORE'
 'VI POSTPAID' 'POS BPCL PETROL PUMP' 'POS INDIAN OIL BANGALORE'
 'UPI-SPOTIFY-IN@AXIS' 'POS DINEOUT' 'COFFEE DAY GLOBAL' 'UPI-NYKAA@AXIS'
 'RAPIDO BIKE TAXI' 'DISNEY HOTSTAR' 'NEFT ZERODHA BROKING'
 'POS EMPIRE RESTAURANT' 'ATM-WDL-SBI-5715' 'POS MEGHANA FOODS'
 'UPI-IOC9075@HDFCBANK' 'FKART INTRNET' 'UPI-BIGBASKET@HDFCBANK'
 'MYNTRA-ORDER' 'BUNDL TECH-INSTAMART' 'MYNTRA DESIGNS' 'POS MYNTRA'
 'POS DMART BANGALORE' 'BIGBASKET BANGALORE' 

In [34]:
vendor_keywords.update({
    # Food delivery and quick commerce
    "Swiggy Instamart": [
        "SWIGGY-INSTAMART",
        "BUNDL TECH-INSTAMART",
        "INSTAMART"
    ],
    "Swiggy": ["SWIGGY", "BUNDL TECH"],
    "BigBasket": [
        "BIGBASKET",
        "INNOVATIVE RETAIL"
    ],
    "Zepto": ["ZEPTO", "KIRANAKART"],

    # Transport
    "Ola": ["OLA CABS", "OLACABS", "OLA-PRIME",
            "ANI TECHNOLOGIES"],
    "Rapido": ["RAPIDO", "ROPPEN"],

    # Shopping
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],
    "DMart": ["DMART", "AVENUE SUPERMARTS"],

    # Cafe and restaurants
    "Cafe Coffee Day": [
        "CAFE COFFEE DAY",
        "COFFEE DAY GLOBAL",
        "UPI-CCD"
    ],
    "Third Wave Coffee": [
        "THIRDWAVE",
        "THIRD WAVE",
        "TWC INDIA"
    ],
    "Empire Restaurant": ["EMPIRE RESTAURANT"],
    "Meghana Foods": ["MEGHANA FOODS"],
    "Restaurant (Other)": [
        "BANGALORE RESTAURANT",
        "UPI-RESTAURANT",
        "DINEOUT"
    ],

    # Subscriptions
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Disney Hotstar": [
        "HOTSTAR",
        "DISNEY",
        "STAR INDIA PVT LTD"
    ],

    # Investments
    "Zerodha": ["ZERODHA"],

    # Utilities and telecom
    "Jio": ["JIO", "RELIANCE JIO"],
    "Airtel": ["AIRTEL", "BHARTI AIRTEL"],
    "Vi": ["VI POSTPAID", "UPI-VI-", "VODAFONE IDEA"],
    "BWSSB": ["BWSSB"],

    # Fuel
    "HP Petrol": ["HP PETROL"],
    "BPCL": ["BPCL"],
    "Indian Oil": ["INDIAN OIL", "UPI-IOC"],

    # Entertainment
    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS MOVIE",
        "BIGTREE ENTERTAINMENT"
    ],

    # Personal transfers
    "Ankit": ["UPI-ANKIT"],
    "Vikas": ["UPI-VIKAS"],
    "Priya": ["UPI-PRIYA"],
    "Neha": ["UPI-NEHA"],
    "Sneha": ["UPI-SNEHA"],
    "Karan": ["UPI-KARAN"],
    "Landlord": ["IMPS-RENT-LANDLORD"],

    # Cash withdrawals
    "ATM Withdrawal": ["ATM-WDL"]
})

In [35]:
def extract_vendor(description):
    description = str(description).upper()

    matches = []

    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword.upper() in description:
                matches.append((len(keyword), vendor))

    if matches:
        matches.sort(reverse=True)
        return matches[0][1]

    return "Unknown"

In [36]:
df["vendor_clean"] = df["Description"].apply(
    extract_vendor
)

In [37]:
print("Total transactions:", len(df))

print("Unknown transactions:",
      (df["vendor_clean"] == "Unknown").sum())

print("Unique identified vendors:",
      df.loc[
          df["vendor_clean"] != "Unknown",
          "vendor_clean"
      ].nunique())

print("\nRemaining unknown descriptions:")

print(
    df.loc[
        df["vendor_clean"] == "Unknown",
        "Description"
    ].unique()
)

Total transactions: 1310
Unknown transactions: 0
Unique identified vendors: 47

Remaining unknown descriptions:
[]


In [38]:
print("All identified vendors:")

print(
    df["vendor_clean"]
    .value_counts()
    .to_string()
)

All identified vendors:
vendor_clean
Swiggy                176
Zomato                121
Amazon                 86
Uber                   71
Zepto                  71
Swiggy Instamart       67
Ola                    67
Rapido                 55
Blinkit                55
Flipkart               47
Starbucks              42
Restaurant (Other)     39
BMTC                   37
Third Wave Coffee      31
Cafe Coffee Day        26
DMart                  22
Ola Electric           20
Myntra                 20
Nykaa                  19
BigBasket              19
ATM Withdrawal         17
Indian Oil             17
BESCOM                 15
Zerodha                14
Disney Hotstar         13
Empire Restaurant      13
BookMyShow             13
Meghana Foods          12
Netflix                10
Groww                   9
HP Petrol               9
Truffles                9
Spotify                 8
BWSSB                   8
Vi                      8
Airtel                  6
TechCrush Labs          6
J

In [39]:
check = [
    "INSTAMART",
    "BUNDL",
    "OLA ELECTRIC",
    "OLACABS",
    "RAPIDO",
    "RESTAURANT"
]

for keyword in check:
    print("\n", keyword)

    print(
        df.loc[
            df["Description"].str.contains(
                keyword,
                case=False,
                regex=False,
                na=False
            ),
            ["Description", "vendor_clean"]
        ].drop_duplicates().head(10)
    )


 INSTAMART
                     Description      vendor_clean
13              SWIGGY-INSTAMART  Swiggy Instamart
43           INSTAMART BANGALORE  Swiggy Instamart
187         BUNDL TECH-INSTAMART  Swiggy Instamart
234  UPI-SWIGGY-INSTAMART@OKAXIS  Swiggy Instamart
335                POS INSTAMART  Swiggy Instamart

 BUNDL
              Description      vendor_clean
64         BUNDL Tech P L            Swiggy
187  BUNDL TECH-INSTAMART  Swiggy Instamart

 OLA ELECTRIC
     Description  vendor_clean
10  OLA ELECTRIC  Ola Electric

 OLACABS
             Description vendor_clean
38  UPI-OLACABS@HDFCBANK          Ola

 RAPIDO
           Description vendor_clean
60   UPI-RAPIDO@OKAXIS       Rapido
119   RAPIDO BIKE TAXI       Rapido

 RESTAURANT
                   Description        vendor_clean
20       POS SWIGGY-RESTAURANT              Swiggy
49    POS BANGALORE RESTAURANT  Restaurant (Other)
136      POS EMPIRE RESTAURANT   Empire Restaurant
246  UPI-RESTAURANT-7285@PAYTM  Restaurant (O

## Feature 3: Category Tagger

In [40]:
category_map = {
    # Food delivery
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick commerce
    "Swiggy Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",

    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Ola Electric": "Transport",

    # Cafes
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    # Restaurants
    "Restaurant (Other)": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Meghana Foods": "Restaurants",
    "Truffles": "Restaurants",

    # Groceries
    "DMart": "Groceries",
    "BigBasket": "Groceries",

    # Investments
    "Zerodha": "Investments",
    "Groww": "Investments",

    # Subscriptions
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney Hotstar": "Subscriptions",

    # Utilities
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Jio": "Utilities",
    "Airtel": "Utilities",
    "Vi": "Utilities",

    # Fuel
    "Indian Oil": "Fuel",
    "HP Petrol": "Fuel",
    "BPCL": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",

    # Personal transfers
    "Aman": "Personal Transfer",
    "Ankit": "Personal Transfer",
    "Vikas": "Personal Transfer",
    "Priya": "Personal Transfer",
    "Neha": "Personal Transfer",
    "Sneha": "Personal Transfer",
    "Karan": "Personal Transfer",

    # Other transactions
    "Landlord": "Rent",
    "ATM Withdrawal": "Cash Withdrawal",
    "TechCrush Labs": "Salary"
}

In [41]:
df["category"] = df["vendor_clean"].map(category_map)

df[["Description", "vendor_clean", "category"]].head(20)

,Description,vendor_clean,category
0,AMAZON SELLER SVCS,Amazon,E-commerce
1,BHIM-BMTC,BMTC,Transport
2,NEFT-TECHCRUSH LABS-SALARY MAY24,TechCrush Labs,Salary
3,UPI-AMAN-8934@OKAXIS,Aman,Personal Transfer
4,BHIM-BLINKIT,Blinkit,Quick Commerce
5,BHIM ZEPTO,Zepto,Quick Commerce
6,UPI-UBER-2426@HDFCBANK,Uber,Transport
7,BHIM-BLINKIT,Blinkit,Quick Commerce
8,POS SWIGGY BANGALORE,Swiggy,Food Delivery
9,UPI-GROWWPAY@HDFCBANK,Groww,Investments


In [42]:
print("Missing categories:", df["category"].isna().sum())

print("Category counts:")
print(df["category"].value_counts())

print("Unmapped vendors:")
print(
    df.loc[
        df["category"].isna(),
        "vendor_clean"
    ].unique()
)

Missing categories: 0
Category counts:
category
Food Delivery        297
Transport            250
Quick Commerce       193
E-commerce           172
Cafe                  99
Restaurants           73
Utilities             43
Groceries             41
Subscriptions         31
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Salary                 6
Rent                   6
Name: count, dtype: int64
Unmapped vendors:
[]


## Feature 4: Spending Overview

In [43]:
total_credits = df.loc[
    df["Type"] == "Credit", "Amount"
].sum()

total_debits = df.loc[
    df["Type"] == "Debit", "Amount"
].sum()

print("Total Credits: ₹", round(total_credits, 2))
print("Total Debits: ₹", round(total_debits, 2))

Total Credits: ₹ 509774.0
Total Debits: ₹ 1678901.0


In [44]:
net_savings = total_credits - total_debits

if total_credits > 0:
    savings_rate = (
        net_savings / total_credits
    ) * 100
else:
    savings_rate = 0

print("Net Savings: ₹", round(net_savings, 2))
print("Savings Rate:", round(savings_rate, 2), "%")

Net Savings: ₹ -1169127.0
Savings Rate: -229.34 %


In [45]:
debits = df[df["Type"] == "Debit"]

category_spending = (
    debits.groupby("category")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 5 Spending Categories:")
print(category_spending.head(5))

Top 5 Spending Categories:
category
E-commerce       603877.0
Investments      248160.0
Food Delivery    129054.0
Restaurants      117737.0
Rent             108000.0
Name: Amount, dtype: float64


In [46]:
vendor_spending = (
    debits.groupby("vendor_clean")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 5 Vendors:")
print(vendor_spending.head(5))

Top 5 Vendors:
vendor_clean
Amazon      328530.0
Zerodha     210000.0
Flipkart    177510.0
Landlord    108000.0
Swiggy       73738.0
Name: Amount, dtype: float64


In [47]:
print("===== SPENDING OVERVIEW =====")

print("Total Credits: ₹", round(total_credits, 2))
print("Total Debits: ₹", round(total_debits, 2))
print("Net Savings: ₹", round(net_savings, 2))
print("Savings Rate:", round(savings_rate, 2), "%")

print("\nTop 5 Spending Categories:")
print(category_spending.head(5))

print("\nTop 5 Vendors:")
print(vendor_spending.head(5))

===== SPENDING OVERVIEW =====
Total Credits: ₹ 509774.0
Total Debits: ₹ 1678901.0
Net Savings: ₹ -1169127.0
Savings Rate: -229.34 %

Top 5 Spending Categories:
category
E-commerce       603877.0
Investments      248160.0
Food Delivery    129054.0
Restaurants      117737.0
Rent             108000.0
Name: Amount, dtype: float64

Top 5 Vendors:
vendor_clean
Amazon      328530.0
Zerodha     210000.0
Flipkart    177510.0
Landlord    108000.0
Swiggy       73738.0
Name: Amount, dtype: float64


## Feature 5: Monthly Spending Analysis

In [48]:
df["Month"] = df["Date"].dt.to_period("M")

print(df[["Date", "Month"]].head(10))

        Date    Month
0 2024-01-01  2024-01
1 2024-01-01  2024-01
2 2024-01-01  2024-01
3 2024-01-01  2024-01
4 2024-01-01  2024-01
5 2024-01-01  2024-01
6 2024-01-01  2024-01
7 2024-01-01  2024-01
8 2024-01-02  2024-01
9 2024-01-02  2024-01


In [49]:
monthly_spending = (
    df[df["Type"] == "Debit"]
    .groupby("Month")["Amount"]
    .sum()
    .sort_index()
)

print("Monthly Spending:")
print(monthly_spending)

Monthly Spending:
Month
2024-01    290767.0
2024-02    234071.0
2024-03    330458.0
2024-04    251382.0
2024-05    277306.0
2024-06    294917.0
Freq: M, Name: Amount, dtype: float64


In [50]:
highest_month = monthly_spending.idxmax()
highest_amount = monthly_spending.max()

print("Highest-spending month:", highest_month)
print("Amount: ₹", round(highest_amount, 2))

Highest-spending month: 2024-03
Amount: ₹ 330458.0


In [51]:
monthly_category = df[
    df["Type"] == "Debit"
].pivot_table(
    index="category",
    columns="Month",
    values="Amount",
    aggfunc="sum",
    fill_value=0
)

print(monthly_category.to_string())

Month              2024-01  2024-02   2024-03  2024-04  2024-05   2024-06
category                                                                 
Cafe                3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         98623.0  94011.0  108215.0  69219.0  95776.0  138033.0
Entertainment       1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery      20890.0  21452.0   20850.0  23054.0  22167.0   20641.0
Fuel               30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries          17649.0   8571.0    6289.0  11748.0   9718.0    5432.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer   7852.0   4285.0    4275.0    663.0   3412.0    4112.0
Quick Commerce     12797.0  17465.0   17979.0  16572.0  15188.0   15666.0
Rent               18000.0  18000.0   18000.0  18000.0  18000.0   18000.0
Restaurants        16320.0  21772.0   

In [52]:
monthly_change = monthly_category.diff(axis=1)

print("Month-to-month changes:")
print(monthly_change.to_string())

Month-to-month changes:
Month              2024-01  2024-02  2024-03  2024-04  2024-05  2024-06
category                                                               
Cafe                   NaN    583.0   1175.0   1116.0   -896.0    134.0
Cash Withdrawal        NaN   3000.0   3000.0  -2500.0   2500.0   9000.0
E-commerce             NaN  -4612.0  14204.0 -38996.0  26557.0  42257.0
Entertainment          NaN   -789.0   1944.0   -194.0  -2224.0   1914.0
Food Delivery          NaN    562.0   -602.0   2204.0   -887.0  -1526.0
Fuel                   NaN -28243.0  24085.0  -7446.0  -9580.0  -6256.0
Groceries              NaN  -9078.0  -2282.0   5459.0  -2030.0  -4286.0
Investments            NaN -23432.0  53644.0 -14518.0  -5498.0 -25298.0
Personal Transfer      NaN  -3567.0    -10.0  -3612.0   2749.0    700.0
Quick Commerce         NaN   4668.0    514.0  -1407.0  -1384.0    478.0
Rent                   NaN      0.0      0.0      0.0      0.0      0.0
Restaurants            NaN   5452.0   65

In [53]:
changes = monthly_change.stack()

print("Largest spending increase:")
print(changes.idxmax())
print("Amount: ₹", changes.max())

print("\nLargest spending decrease:")
print(changes.idxmin())
print("Amount: ₹", changes.min())

Largest spending increase:
('Investments', Period('2024-03', 'M'))
Amount: ₹ 53644.0

Largest spending decrease:
('E-commerce', Period('2024-04', 'M'))
Amount: ₹ -38996.0


## Feature 6: Time-of-Day Spending Analysis

In [54]:
print("Time column data type:", df["Time"].dtype)

print("\nSample transaction times:")
print(df["Time"].head(15).to_string(index=False))

print("\nMissing time values:")
print(df["Time"].isna().sum())

Time column data type: object

Sample transaction times:
03:11
05:44
09:35
14:07
14:23
14:48
14:50
21:44
05:18
06:55
07:26
07:39
11:29
12:35
14:01

Missing time values:
0


In [55]:
df["Hour"] = df["Time"].str.split(":").str[0].astype(int)

print(df[["Time", "Hour"]].head(15))
print("\nInvalid hours:", (~df["Hour"].between(0, 23)).sum())

     Time  Hour
0   03:11     3
1   05:44     5
2   09:35     9
3   14:07    14
4   14:23    14
5   14:48    14
6   14:50    14
7   21:44    21
8   05:18     5
9   06:55     6
10  07:26     7
11  07:39     7
12  11:29    11
13  12:35    12
14  14:01    14

Invalid hours: 0


In [56]:
def get_time_period(hour):
    if 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["Time_Period"] = df["Hour"].apply(get_time_period)

print(df[["Time", "Hour", "Time_Period"]].head(15))

print("\nTransactions by time period:")
print(df["Time_Period"].value_counts())

     Time  Hour Time_Period
0   03:11     3       Night
1   05:44     5       Night
2   09:35     9     Morning
3   14:07    14   Afternoon
4   14:23    14   Afternoon
5   14:48    14   Afternoon
6   14:50    14   Afternoon
7   21:44    21       Night
8   05:18     5       Night
9   06:55     6     Morning
10  07:26     7     Morning
11  07:39     7     Morning
12  11:29    11     Morning
13  12:35    12   Afternoon
14  14:01    14   Afternoon

Transactions by time period:
Time_Period
Evening      340
Night        336
Morning      320
Afternoon    314
Name: count, dtype: int64


In [57]:
time_spending = (
    df[df["Type"] == "Debit"]
    .groupby("Time_Period")["Amount"]
    .agg(["count", "sum", "mean"])
    .reindex(["Morning", "Afternoon", "Evening", "Night"])
)

time_spending.columns = [
    "Transactions",
    "Total_Spending",
    "Average_Transaction"
]

print(time_spending.round(2))

print("\nTotal spending:", time_spending["Total_Spending"].sum())

             Transactions  Total_Spending  Average_Transaction
Time_Period                                                   
Morning               314        467459.0              1488.72
Afternoon             314        452169.0              1440.03
Evening               340        363884.0              1070.25
Night                 336        395389.0              1176.75

Total spending: 1678901.0


In [58]:
food_orders = df[
    (df["Type"] == "Debit") &
    (df["category"] == "Food Delivery")
].copy()

late_night_orders = food_orders[
    (food_orders["Hour"] >= 21) |
    (food_orders["Hour"] < 3)
]

late_night_percentage = (
    len(late_night_orders) / len(food_orders) * 100
    if len(food_orders) > 0 else 0
)

print("Total food delivery orders:", len(food_orders))
print("Late-night food delivery orders:", len(late_night_orders))
print("Late-night percentage:", round(late_night_percentage, 2), "%")

print(
    "Late-night food delivery spending: ₹",
    round(late_night_orders["Amount"].sum(), 2)
)

Total food delivery orders: 297
Late-night food delivery orders: 63
Late-night percentage: 21.21 %
Late-night food delivery spending: ₹ 25685.0


## Feature 7: Anomaly Detection

In [59]:
debits = df[df["Type"] == "Debit"].copy()

category_stats = (
    debits.groupby("category")["Amount"]
    .agg(["mean", "std", "count"])
    .round(2)
)

print(category_stats.to_string())

                       mean      std  count
category                                   
Cafe                 317.63    86.31     99
Cash Withdrawal     2676.47  1853.55     17
E-commerce          3510.91  4522.13    172
Entertainment        637.92   262.66     13
Food Delivery        434.53   147.50    297
Fuel                3189.39  3213.35     28
Groceries           1448.95   639.72     41
Investments        10789.57  5376.35     23
Personal Transfer   1366.61   816.11     18
Quick Commerce       495.68   199.09    193
Rent               18000.00     0.00      6
Restaurants         1612.84  1742.80     73
Subscriptions        595.84   442.89     31
Transport            229.90   147.40    250
Utilities            974.74   720.17     43


In [60]:
debits["Category_Mean"] = (
    debits.groupby("category")["Amount"].transform("mean")
)

debits["Category_Std"] = (
    debits.groupby("category")["Amount"].transform("std")
)

debits["Z_Score"] = (
    (debits["Amount"] - debits["Category_Mean"])
    / debits["Category_Std"].replace(0, float("nan"))
)

print(
    debits[
        ["category", "Amount", "Category_Mean",
         "Category_Std", "Z_Score"]
    ].head(10).round(2).to_string(index=False)
)

print("\nUndefined z-scores:", debits["Z_Score"].isna().sum())

         category  Amount  Category_Mean  Category_Std  Z_Score
       E-commerce  2462.0        3510.91       4522.13    -0.23
        Transport    50.0         229.90        147.40    -1.22
Personal Transfer  1828.0        1366.61        816.11     0.57
   Quick Commerce   270.0         495.68        199.09    -1.13
   Quick Commerce   625.0         495.68        199.09     0.65
        Transport   148.0         229.90        147.40    -0.56
   Quick Commerce   482.0         495.68        199.09    -0.07
    Food Delivery   537.0         434.53        147.50     0.69
      Investments  3956.0       10789.57       5376.35    -1.27
        Transport   350.0         229.90        147.40     0.81

Undefined z-scores: 6


In [61]:
anomalies = (
    debits[debits["Z_Score"] > 2]
    .sort_values("Z_Score", ascending=False)
)

print("Number of anomalies:", len(anomalies))

print(
    anomalies[
        ["Date", "vendor_clean", "category",
         "Amount", "Z_Score"]
    ].round(2).to_string(index=False)
)


Number of anomalies: 24
      Date       vendor_clean    category  Amount  Z_Score
2024-06-26             Amazon  E-commerce 22008.0     4.09
2024-02-07             Amazon  E-commerce 21986.0     4.09
2024-02-26 Restaurant (Other) Restaurants  8383.0     3.88
2024-03-05             Amazon  E-commerce 19917.0     3.63
2024-06-22 Restaurant (Other) Restaurants  7935.0     3.63
2024-03-31      Meghana Foods Restaurants  7931.0     3.63
2024-03-04           Truffles Restaurants  7441.0     3.34
2024-01-07 Restaurant (Other) Restaurants  7314.0     3.27
2024-03-02             Amazon  E-commerce 18273.0     3.26
2024-05-27           Flipkart  E-commerce 17831.0     3.17
2024-06-25             Amazon  E-commerce 17504.0     3.09
2024-06-03           Flipkart  E-commerce 16369.0     2.84
2024-01-24           Flipkart  E-commerce 16260.0     2.82
2024-06-27           Flipkart  E-commerce 15955.0     2.75
2024-03-31             Amazon  E-commerce 15953.0     2.75
2024-04-30             Amazon  E

In [62]:
anomaly_summary = (
    anomalies.groupby("category")
    .agg(
        Anomaly_Count=("Amount", "count"),
        Total_Amount=("Amount", "sum"),
        Highest_Z_Score=("Z_Score", "max")
    )
    .sort_values("Anomaly_Count", ascending=False)
)

print(anomaly_summary.round(2).to_string())


             Anomaly_Count  Total_Amount  Highest_Z_Score
category                                                 
E-commerce              15      254211.0             4.09
Restaurants              5       39004.0             3.88
Fuel                     2       20913.0             2.28
Groceries                2        5557.0             2.09


## Feature 8: Spending Archetypes

In [63]:
category_percentages = (
    category_spending / total_debits * 100
).sort_values(ascending=False)

print("Percentage of total debits by category:")
print(category_percentages.round(2).to_string())

food_related_percentage = (
    category_spending.reindex(
        ["Food Delivery", "Restaurants", "Cafe"],
        fill_value=0
    ).sum() / total_debits * 100
)

print(
    "\nCombined food-related spending:",
    round(food_related_percentage, 2), "%"
)

Percentage of total debits by category:
category
E-commerce           35.97
Investments          14.78
Food Delivery         7.69
Restaurants           7.01
Rent                  6.43
Quick Commerce        5.70
Fuel                  5.32
Groceries             3.54
Transport             3.42
Cash Withdrawal       2.71
Utilities             2.50
Cafe                  1.87
Personal Transfer     1.47
Subscriptions         1.10
Entertainment         0.49

Combined food-related spending: 16.57 %


In [64]:
archetypes = []

if category_percentages.get("E-commerce", 0) > 15:
    archetypes.append("Shopaholic")

if food_related_percentage > 25:
    archetypes.append("Foodie")

if category_percentages.get("Investments", 0) > 15:
    archetypes.append("Investor")

if category_percentages.get("Quick Commerce", 0) > 15:
    archetypes.append("Quick Commerce Enthusiast")

if category_percentages.get("Transport", 0) > 10:
    archetypes.append("Cab Commuter")

if late_night_percentage > 50:
    archetypes.append("Late-Night Snacker")

if savings_rate < 10:
    archetypes.append("YOLO Spender")

if savings_rate > 40:
    archetypes.append("Disciplined Saver")

subscription_vendors = df.loc[
    (df["Type"] == "Debit") &
    (df["category"] == "Subscriptions"),
    "vendor_clean"
].nunique()

if subscription_vendors >= 5:
    archetypes.append("Subscription Lover")

print("Distinct subscription vendors:", subscription_vendors)
print("\nAssigned spending archetypes:")
for archetype in archetypes:
    print("-", archetype)

Distinct subscription vendors: 3

Assigned spending archetypes:
- Shopaholic
- YOLO Spender


## SpendDNA Report

In [65]:
print("=" * 55)
print("                    SPENDDNA")
print("          PERSONAL SPENDING ANALYSIS")
print("=" * 55)

print("\nANALYSIS PERIOD: January – June 2024")
print("TOTAL TRANSACTIONS:", len(df))

print("\n" + "-" * 55)
print("1. SPENDING OVERVIEW")
print("-" * 55)

print(f"Total Credits : ₹{total_credits:,.2f}")
print(f"Total Debits  : ₹{total_debits:,.2f}")
print(f"Net Cash Flow : ₹{net_savings:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")

print("\nTop 5 Spending Categories:")
for category, amount in category_spending.head(5).items():
    print(f"  {category:<20} ₹{amount:>12,.2f}")

print("\nTop 5 Vendors:")
for vendor, amount in vendor_spending.head(5).items():
    print(f"  {vendor:<20} ₹{amount:>12,.2f}")

                    SPENDDNA
          PERSONAL SPENDING ANALYSIS

ANALYSIS PERIOD: January – June 2024
TOTAL TRANSACTIONS: 1310

-------------------------------------------------------
1. SPENDING OVERVIEW
-------------------------------------------------------
Total Credits : ₹509,774.00
Total Debits  : ₹1,678,901.00
Net Cash Flow : ₹-1,169,127.00
Savings Rate  : -229.34%

Top 5 Spending Categories:
  E-commerce           ₹  603,877.00
  Investments          ₹  248,160.00
  Food Delivery        ₹  129,054.00
  Restaurants          ₹  117,737.00
  Rent                 ₹  108,000.00

Top 5 Vendors:
  Amazon               ₹  328,530.00
  Zerodha              ₹  210,000.00
  Flipkart             ₹  177,510.00
  Landlord             ₹  108,000.00
  Swiggy               ₹   73,738.00


In [66]:
# Monthly Food Delivery spending
food_monthly = (
    df[
        (df["Type"] == "Debit") &
        (df["category"] == "Food Delivery")
    ]
    .groupby("Month")["Amount"]
    .sum()
    .sort_index()
)

print("FOOD DELIVERY MONTHLY TREND")
print(food_monthly)

# Transaction counts for the top five vendors
vendor_counts = (
    df[df["Type"] == "Debit"]
    .groupby("vendor_clean")
    .size()
)

print("\nTOP VENDOR COUNTS")
for vendor in vendor_spending.head(5).index:
    print(vendor, ":", vendor_counts[vendor], "transactions")

# Peak transaction hours
print("\nPEAK HOURS BY CATEGORY")
for category in ["Food Delivery", "Cafe", "Quick Commerce"]:
    subset = df[
        (df["Type"] == "Debit") &
        (df["category"] == category)
    ]

    hourly_counts = subset.groupby("Hour").size()
    peak_hour = hourly_counts.idxmax()

    print(
        category, ":",
        f"{peak_hour:02d}:00–{(peak_hour + 1) % 24:02d}:00",
        f"({hourly_counts.max()} transactions)"
    )

FOOD DELIVERY MONTHLY TREND
Month
2024-01    20890.0
2024-02    21452.0
2024-03    20850.0
2024-04    23054.0
2024-05    22167.0
2024-06    20641.0
Freq: M, Name: Amount, dtype: float64

TOP VENDOR COUNTS
Amazon : 86 transactions
Zerodha : 14 transactions
Flipkart : 47 transactions
Landlord : 6 transactions
Swiggy : 176 transactions

PEAK HOURS BY CATEGORY
Food Delivery : 20:00–21:00 (36 transactions)
Cafe : 10:00–11:00 (13 transactions)
Quick Commerce : 20:00–21:00 (20 transactions)


In [67]:

# FINAL SPENDDNA PRINTED REPORT

debit_df = df[df["Type"] == "Debit"].copy()

# A simple text bar for the printed report
def make_bar(value, maximum, width=20):
    if maximum == 0:
        return ""
    length = round(value / maximum * width)
    return "#" * length

# Format negative currency as -Rs. 1,000
def rupees(amount):
    sign = "-" if amount < 0 else ""
    return f"{sign}Rs. {abs(amount):,.0f}"

line = "=" * 65

print(line)
print("                 SpendDNA REPORT - RAHUL SHARMA")
print(f"       6 months  -  {len(df):,} transactions  -  Jan to Jun 2024")
print(line)

# EXECUTIVE SUMMARY
print("\nEXECUTIVE SUMMARY")
print(f"  Total credits   : {rupees(total_credits)}")
print(f"  Total debits    : {rupees(total_debits)}")
print(f"  Net change      : {rupees(net_savings)}")
print(f"  Savings rate    : {savings_rate:.2f}%")
print(f"  Transactions    : {len(df):,}")
print(f"  Unique vendors  : {df['vendor_clean'].nunique()}")

# TOP CATEGORIES
print("\nTOP CATEGORIES (% of debit total)")
top_categories = category_spending.head(5)
maximum = top_categories.max()

for category, amount in top_categories.items():
    percentage = amount / total_debits * 100
    bar = make_bar(amount, maximum)
    print(
        f"  {category:<17} {bar:<20} "
        f"{percentage:5.1f}%  {rupees(amount)}"
    )

# TOP VENDORS
print("\nTOP VENDORS")
vendor_counts = debit_df.groupby("vendor_clean").size()

for vendor, amount in vendor_spending.head(5).items():
    count = vendor_counts[vendor]
    print(
        f"  {vendor:<18} {rupees(amount):>16} "
        f"({count} transactions)"
    )

# TIME-OF-DAY PATTERNS
print("\nTIME-OF-DAY PATTERNS")

for category in ["Food Delivery", "Cafe", "Quick Commerce"]:
    subset = debit_df[debit_df["category"] == category]
    hourly_counts = subset.groupby("Hour").size()
    peak_hour = int(hourly_counts.idxmax())
    peak_count = int(hourly_counts.max())

    print(
        f"  {category:<17} peaks: "
        f"{peak_hour:02d}:00 - {(peak_hour + 1) % 24:02d}:00 "
        f"({peak_count} transactions)"
    )

print(
    f"  Late-night food orders: {len(late_night_orders)} "
    f"of {len(food_orders)} ({late_night_percentage:.1f}%)"
)

# MONTHLY FOOD DELIVERY TREND
print("\nMONTHLY TREND (Food Delivery)")

food_monthly = (
    debit_df[debit_df["category"] == "Food Delivery"]
    .groupby("Month")["Amount"]
    .sum()
    .sort_index()
)

for month, amount in food_monthly.items():
    bar = make_bar(amount, food_monthly.max())
    print(
        f"  {month.strftime('%b')}  "
        f"{rupees(amount):>12}  {bar}"
    )

# TOP ANOMALIES
print("\nTOP ANOMALIES (more than 2 std dev above category mean)")
print(f"  Total flagged transactions: {len(anomalies)}")

for _, transaction in anomalies.head(3).iterrows():
    date_text = transaction["Date"].strftime("%d %b")
    vendor = transaction["vendor_clean"]
    amount = rupees(transaction["Amount"])
    z_score = transaction["Z_Score"]

    print(
        f"  {date_text} - {vendor:<18} "
        f"{amount:>13}  (z={z_score:.2f})"
    )

# SPENDING ARCHETYPES
print("\nRAHUL'S SPENDING ARCHETYPES")

for archetype in archetypes:
    if archetype == "Shopaholic":
        detail = (
            f"{category_percentages['E-commerce']:.1f}% "
            "on E-commerce"
        )
    elif archetype == "YOLO Spender":
        detail = f"calculated savings rate {savings_rate:.1f}%"
    else:
        detail = "project threshold met"

    print(f"  -> THE {archetype.upper()} ({detail})")

# THREE DATA-DRIVEN INSIGHTS
print("\nKEY INSIGHTS")

print(
    f"  1. E-commerce accounts for "
    f"{category_percentages['E-commerce']:.1f}% "
    "of total debits."
)

print(
    f"  2. March has the highest total debits "
    f"({rupees(monthly_spending.max())})."
)

print(
    f"  3. {late_night_percentage:.1f}% of Food Delivery "
    "orders occurred between 9 PM and 3 AM."
)

print("\nNOTE: Net change includes investments, transfers and")
print("cash withdrawals as debits; it is not actual savings.")

print(line)
print("                    END OF REPORT")
print(line)

                 SpendDNA REPORT - RAHUL SHARMA
       6 months  -  1,310 transactions  -  Jan to Jun 2024

EXECUTIVE SUMMARY
  Total credits   : Rs. 509,774
  Total debits    : Rs. 1,678,901
  Net change      : -Rs. 1,169,127
  Savings rate    : -229.34%
  Transactions    : 1,310
  Unique vendors  : 47

TOP CATEGORIES (% of debit total)
  E-commerce        ####################  36.0%  Rs. 603,877
  Investments       ########              14.8%  Rs. 248,160
  Food Delivery     ####                   7.7%  Rs. 129,054
  Restaurants       ####                   7.0%  Rs. 117,737
  Rent              ####                   6.4%  Rs. 108,000

TOP VENDORS
  Amazon                  Rs. 328,530 (86 transactions)
  Zerodha                 Rs. 210,000 (14 transactions)
  Flipkart                Rs. 177,510 (47 transactions)
  Landlord                Rs. 108,000 (6 transactions)
  Swiggy                   Rs. 73,738 (176 transactions)

TIME-OF-DAY PATTERNS
  Food Delivery     peaks: 20:00 - 21:00

In [68]:
%%capture report_output

In [69]:
report_text = report_output.stdout

print(report_text)

with open("SpendDNA_Report.txt", "w", encoding="utf-8") as file:
    file.write(report_text)

print("\nReport saved successfully!")



Report saved successfully!


In [70]:
from google.colab import files

files.download("SpendDNA_Report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [71]:
readme_text = """# SpendDNA — Personal Spending Analyzer

SpendDNA analyzes six months of synthetic transaction data
using Python, Pandas, and NumPy.

## Features

1. Transaction parsing and cleaning
2. Vendor extraction
3. Category tagging
4. Spending overview
5. Monthly spending trends
6. Time-of-day analysis
7. Category-wise z-score anomaly detection
8. Spending archetype detection

## Dataset

January–June 2024 | 1,310 transactions | 47 vendors

## Key Results

- Total credits: Rs. 509,774
- Total debits: Rs. 1,678,901
- Highest-spending month: March 2024
- Largest category: E-commerce (35.97%)
- Anomalies detected: 24
- Archetypes: Shopaholic, YOLO Spender

## How to Run

1. Open SpendDNA.ipynb in Google Colab.
2. Upload rahul_transactions.csv.
3. Run all notebook cells in order.
4. Review SpendDNA_Report.txt for the final report.

## Technologies

Python, Pandas, NumPy, Google Colab

## Note

The dataset is synthetic. Net cash flow includes
investments, transfers, and cash withdrawals as debits.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_text)

print("README.md saved successfully!")

README.md saved successfully!
